In [12]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [10]:
!ls StoryTTS/LianLiru_ZSDFS/llr/episode1/LianLiru-ZSDFS-episode1-seg001.wav

StoryTTS/LianLiru_ZSDFS/llr/episode1/LianLiru-ZSDFS-episode1-seg001.wav


In [4]:
with open('StoryTTS/transcript') as fopen:
    t = fopen.read().split('\n')
len(t)

33108

In [24]:
import re
from pathlib import Path

def meta_to_path(meta_id: str) -> Path:
    """
    Convert metadata ID like 'LianLiru-ZSDFS-episode001-seg001'
    into file path 'StoryTTS/LianLiru_ZSDFS/llr/episode1/LianLiru-ZSDFS-episode1-seg001.wav'
    """
    # Extract episode number
    m = re.search(r"episode0*(\d+)", meta_id)
    if not m:
        raise ValueError(f"Cannot parse episode from: {meta_id}")
    episode_num = int(m.group(1))  # remove leading zeros

    # Replace the metadata ID to use episode<num> (no leading zeros)
    norm_id = re.sub(r"episode0*\d+", f"episode{episode_num}", meta_id)

    # Build path
    return f"StoryTTS/LianLiru_ZSDFS/llr/episode{episode_num}/{norm_id}.wav"

# Example usage
meta_id = "LianLiru-ZSDFS-episode001-seg001"
file_path = meta_to_path(meta_id)

print(file_path)

StoryTTS/LianLiru_ZSDFS/llr/episode1/LianLiru-ZSDFS-episode1-seg001.wav


In [20]:
d = []
for t_ in t:
    f = t_.split()
    d.append((f[0], ' '.join(f[1:])))
len(d)

33108

In [27]:
!mkdir StoryTTS_audio

In [29]:
def loop(rows):
    rows, _ = rows
    data = []
    for row in tqdm(rows):
        try:
            f = meta_to_path(row[0])
            t = row[1].strip()
            if len(t) < 2:
                continue

            audio_np, sr = sf.read(f)
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue

            audio_filename = f.replace('/', '_').replace('.wav', '.mp3')
            audio_filename = os.path.join('StoryTTS_audio', audio_filename)
                
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': "StoryTTS"
            })
        except Exception as e:
            print(e)
            pass
    return data

In [30]:
data = loop((d[:10], 0))

100%|██████████| 10/10 [00:00<00:00, 19.26it/s]


In [32]:
data = multiprocessing(d, loop, cores = 20)

 17%|█▋        | 279/1655 [00:14<00:45, 30.15it/s]

Error opening 'StoryTTS/LianLiru_ZSDFS/llr/episode20/LianLiru-ZSDFS-episode20-seg101“.wav': System error.

 19%|█▉        | 313/1655 [00:14<01:04, 20.70it/s]

 93%|█████████▎| 1531/1655 [01:03<00:05, 21.31it/s]

Error opening 'StoryTTS/LianLiru_ZSDFS/llr/episode128/LianLiru-ZSDFS-episode128-seg1960.wav': System error.


100%|██████████| 1655/1655 [01:25<00:00, 19.33it/s]


In [33]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'StoryTTS_audio/StoryTTS_LianLiru_ZSDFS_llr_episode1_LianLiru-ZSDFS-episode1-seg001.mp3',
 'text': '秦朝末年，沛公刘邦在芒荡山斩白蛇，揭竿而起。三载亡秦，五年破楚，打下了汉室天下。',
 'speaker': 'StoryTTS'}

In [34]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'StoryTTS')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 73.60ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  95%|█████████▌| 1.74MB / 1.82MB, 8.69MB/s  
Processing Files (1 / 1): 100%|██████████| 1.82MB / 1.82MB, 5.41MB/s  
Processing Files (1 / 1): 100%|██████████| 1.82MB / 1.82MB, 4.56MB/s  
New Data Upload: 100%|██████████| 1.82MB / 1.82MB, 4.56MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.11 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/dac38872cddcdf6eae7744e0fc07d35001f5f886', commit_message='Upload dataset', commit_description='', oid='dac38872cddcdf6eae7744e0fc07d35001f5f886', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [35]:
audio_files = [d['audio_filename'] for d in data]

with open('StoryTTS-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [38]:
folders = glob('StoryTTS_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

StoryTTS_audio
StoryTTS_audio_neucodec


In [40]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('StoryTTS_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   3%|▎         | 39.3MB / 1.18GB,   ???B/s  
Processing Files (0 / 1):  15%|█▍        |  177MB / 1.18GB,  690MB/s  
Processing Files (0 / 1):  26%|██▌       |  310MB / 1.18GB,  677MB/s  
Processing Files (0 / 1):  37%|███▋      |  441MB / 1.18GB,  669MB/s  
Processing Files (0 / 1):  45%|████▍     |  531MB / 1.18GB,  614MB/s  
Processing Files (0 / 1):  53%|█████▎    |  632MB / 1.18GB,  592MB/s  
Processing Files (0 / 1):  62%|██████▏   |  729MB / 1.18GB,  575MB/s  
Processing Files (0 / 1):  75%|███████▍  |  883MB / 1.18GB,  603MB/s  
Processing Files (0 / 1):  86%|████████▌ | 1.02GB / 1.18GB,  612MB/s  
Processing Files (0 / 1):  98%|█████████▊| 1.16GB / 1.18GB,  624MB/s  
Processing Files (0 / 1): 100%|█████████▉| 1.18GB / 1.18GB,  571MB/s  
Processing Files (0 / 1): 100%|█████████▉| 1.18GB / 1.18GB,  519MB/s  
Processing Files (0 / 1): 100%|█████████▉| 1.18GB / 1.18GB,  477MB/s  
Processing